# Notebook 1 | Parameter Discovery
## GeoINquire Workshop · Messina, April 2026

**Purpose:** Query the EFEHR/SHARE hazard web service to find out *what is available* for a given location — which hazard models, intensity measures, soil types, and return periods you can access before plotting anything.

Think of this notebook as the *menu* step: before ordering, you check what the kitchen has.

### What you will learn
- How the EFEHR SHARE API is structured (a cascading set of XML queries)
- Which models cover Italy and Southern Europe
- How Intensity Measure Types (IMTs), soil categories and aggregation levels are encoded
- How return periods are stored — and why the ESHM20 format can be confusing

### Output
A `hazard_config.yaml` file that `Notebook 2` reads automatically. **Run this notebook first** whenever you change the site coordinates.

---
### Background: The SHARE/EFEHR API

The European Seismic Hazard Model (ESHM) results are served by a REST API hosted at `appsrvr.share-eu.org`. Queries are HTTP GET requests with URL parameters; responses are XML documents. The API is hierarchical:

```
Level 1  → ?lat=&lon=            → available Models at this location
Level 2  → + &id=                → available IMTs for this Model
Level 3  → + &imt=               → available Soil Types for this IMT
Level 4  → + &soiltype=          → available Aggregations (mean, percentiles)
Level 5  → spectra endpoint      → available Probabilities of Exceedance
```

This notebook walks through all five levels and saves the results to YAML.

---
## Cell 1 — Install dependencies (run once)

In [ ]:
# Run this cell once to make sure the required packages are installed
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pyyaml', 'requests'])
print('Dependencies OK.')

---
## Cell 2 — Set your site location

**Student exercise:** Change the coordinates below to your site of interest. Some suggestions:

| City | Latitude | Longitude | Why interesting |
|------|----------|-----------|----------------|
| Messina (default) | 38.19 | 15.55 | 1908 earthquake, Calabrian Arc |
| Catania | 37.50 | 15.09 | Mount Etna + crustal seismicity |
| Naples | 40.85 | 14.27 | Campi Flegrei + Southern Apennines |
| L'Aquila | 42.35 | 13.40 | 2009 Mw 6.3 earthquake |
| Bucharest | 44.43 | 26.10 | Vrancea intermediate-depth source |
| Basel | 47.56 | 7.59 | Historical damaging earthquake 1356 |

In [ ]:
# ============================================================
#  CONFIGURATION — edit these values for your site
# ============================================================
LAT = 38.19   # Messina, Sicily
LON = 15.55

BASE_URL    = 'http://appsrvr.share-eu.org:8080/share'
OUTPUT_FILE = 'hazard_config.yaml'

print(f'Site set to: {LAT}°N, {LON}°E')

---
## Cell 3 — Helper functions

These utility functions handle HTTP requests to the API and strip XML namespaces (which can make parsing messy). You do not need to modify these.

In [ ]:
import requests
import xml.etree.ElementTree as ET
import io
import yaml

def get_xml_root(url, verbose=True):
    """Fetch a URL, parse the XML response, and strip namespace prefixes."""
    try:
        if verbose:
            print(f'  → GET {url}')
        response = requests.get(url, timeout=15)
        response.raise_for_status()
        it = ET.iterparse(io.StringIO(response.text))
        for _, el in it:
            _, _, el.tag = el.tag.rpartition('}')
        return it.root
    except Exception as e:
        print(f'  ✗ Error: {e}')
        return None

def annual_rate_to_return_period(prob_str, years_str):
    """
    Convert an annual exceedance probability to an engineering-friendly label.
    
    ESHM20 stores PoEs as annual rates (timespan = 1 year).
    ESHM13 and national models use cumulative probabilities over 50 years.
    """
    import math
    try:
        prob  = float(prob_str)
        years = float(years_str)
        # Annual rate λ from cumulative probability: λ = -ln(1-P)/T
        # If timespan = 1 year, the stored value is already the annual rate
        if years == 1:
            annual_rate = prob
        else:
            annual_rate = -math.log(1 - prob) / years
        return_period = round(1.0 / annual_rate)
        # Compute probability in 50 years for intuition
        prob_50 = round((1 - math.exp(-annual_rate * 50)) * 100, 1)
        return f'{return_period}-yr RP  ({prob_50}% in 50y)'
    except Exception:
        return f'{prob_str} in {years_str} years'

print('Helper functions loaded.')

---
## Cell 4 — Discover all parameters

This cell queries all five API levels and builds a dictionary of everything available for your site. It may take 30–60 seconds depending on network speed — each model triggers several sub-queries.

In [ ]:
def discover_parameters(lat, lon):
    print(f'\n=== Parameter Discovery for {lat}°N, {lon}°E ===')

    config_data = {
        'location': {'latitude': lat, 'longitude': lon},
        'available_models': []
    }

    # --- Level 1: Models available at this location ---
    root_models = get_xml_root(f'{BASE_URL}/curve?lat={lat}&lon={lon}')
    if root_models is None:
        print('No response from server. Check network and URL.')
        return None

    models_found = root_models.findall('model')
    print(f'\nFound {len(models_found)} model(s):')

    for model_elem in models_found:
        m_id   = model_elem.find('id').text
        m_name = model_elem.find('name').text or f'Unnamed (ID={m_id})'
        print(f'\n  Model: {m_name}  [ID={m_id}]')

        model_entry = {'id': m_id, 'name': m_name, 'parameters': {}}

        # --- Level 2: IMTs ---
        root_imts = get_xml_root(f'{BASE_URL}/curve?lat={lat}&lon={lon}&id={m_id}')
        imts = []
        if root_imts:
            for imt in root_imts.findall('imtcode'):
                code = imt.find('code')
                if code is not None:
                    imts.append(code.text)
        model_entry['parameters']['imts'] = imts
        print(f'    IMTs ({len(imts)}): {imts}')

        if not imts:
            config_data['available_models'].append(model_entry)
            continue

        test_imt = imts[0]

        # --- Level 3: Soil types (sampled from first IMT) ---
        root_soil = get_xml_root(f'{BASE_URL}/curve?lat={lat}&lon={lon}&id={m_id}&imt={test_imt}')
        soil_types = []
        if root_soil:
            for st in root_soil.findall('soiltype'):
                t = st.find('type')
                if t is not None:
                    soil_types.append(t.text)
        model_entry['parameters']['soil_types'] = soil_types
        print(f'    Soil types: {soil_types}')

        # --- Level 4: Aggregation types ---
        aggs = []
        if soil_types:
            test_soil = soil_types[0]
            url_agg = (f'{BASE_URL}/curve?lat={lat}&lon={lon}'
                       f'&id={m_id}&imt={test_imt}&soiltype={test_soil}')
            root_agg = get_xml_root(url_agg)
            if root_agg:
                for agg in root_agg.findall('fractile'):
                    t_elem = agg.find('aggregationtype')
                    l_elem = agg.find('aggregationlevel')
                    if t_elem is not None and l_elem is not None:
                        aggs.append({'type': t_elem.text, 'level': l_elem.text})
        model_entry['parameters']['aggregations'] = aggs
        agg_labels = [f"{a['type']} {a['level']}" for a in aggs]
        print(f'    Aggregations: {agg_labels}')

        # --- Level 5: Probabilities of Exceedance (from spectra endpoint) ---
        poes = []
        root_poe = get_xml_root(f'{BASE_URL}/spectra?lat={lat}&lon={lon}&id={m_id}&imt=SA')
        if root_poe:
            for exc in root_poe.findall('exceedance'):
                prob_elem  = exc.find('hmapexceedprob')
                years_elem = exc.find('hmapexceedyears')
                if prob_elem is not None and years_elem is not None:
                    prob  = prob_elem.text
                    years = years_elem.text
                    rp_label = annual_rate_to_return_period(prob, years)
                    poes.append({'prob': prob, 'years': years, 'label': rp_label})

        # Fallback to map endpoint if spectra gave nothing
        if not poes:
            root_poe_map = get_xml_root(f'{BASE_URL}/map?id={m_id}&imt={test_imt}')
            if root_poe_map:
                for exc in root_poe_map.findall('exceedance'):
                    prob_elem  = exc.find('hmapexceedprob')
                    years_elem = exc.find('hmapexceedyears')
                    if prob_elem is not None and years_elem is not None:
                        prob  = prob_elem.text
                        years = years_elem.text
                        rp_label = annual_rate_to_return_period(prob, years)
                        poes.append({'prob': prob, 'years': years, 'label': rp_label})

        # Deduplicate
        seen, unique_poes = set(), []
        for p in poes:
            key = (p['prob'], p['years'])
            if key not in seen:
                seen.add(key)
                unique_poes.append(p)

        model_entry['parameters']['poes'] = unique_poes
        print(f'    PoEs ({len(unique_poes)}): {[p["label"] for p in unique_poes]}')
        config_data['available_models'].append(model_entry)

    return config_data


config = discover_parameters(LAT, LON)
print('\n=== Discovery complete. ===')

---
## Cell 5 — Save configuration to YAML

The `hazard_config.yaml` file is the bridge between this notebook and the Interactive Plotter. You only need to re-run this notebook when you change your site location.

In [ ]:
if config:
    with open(OUTPUT_FILE, 'w') as f:
        yaml.dump(config, f, sort_keys=False, default_flow_style=False)
    print(f'Configuration saved to: {OUTPUT_FILE}')
    print(f'Models available: {[m["name"] for m in config["available_models"]]}')
else:
    print('No configuration to save — check errors above.')

---
## Cell 6 — Summary table (student reference)

This cell prints a readable summary of everything the API offers for your site. Use it as a quick reference when filling in parameters in Notebook 2.

In [ ]:
if config:
    print(f"\n{'='*65}")
    print(f"EFEHR/SHARE Hazard API — Available Parameters")
    print(f"Site: {config['location']['latitude']}°N, {config['location']['longitude']}°E")
    print(f"{'='*65}")

    for m in config['available_models']:
        p = m['parameters']
        print(f"\n  Model : {m['name']}  [ID={m['id']}]")
        print(f"  IMTs  : {', '.join(p.get('imts', []))}")
        print(f"  Soil  : {', '.join(p.get('soil_types', []))}")
        print(f"  Agg.  : {', '.join([a['type']+' '+a['level'] for a in p.get('aggregations',[])])}")
        poe_strs = [poe['label'] for poe in p.get('poes', [])]
        print(f"  PoEs  : {', '.join(poe_strs) if poe_strs else 'None found (will use defaults)'}")
    print(f"\n{'='*65}")
    print("\nNote on aggregations:")
    print("  arithmetic 0.5 = mean of the logic-tree branches")
    print("  ordinal   0.5  = median (50th percentile)")
    print("  ordinal   0.16 = 16th percentile (lower uncertainty bound)")
    print("  ordinal   0.84 = 84th percentile (upper uncertainty bound)")
    print("\nNote on return periods (ESHM20 uses annual rates):")
    print("  475-yr RP  = 10% probability of exceedance in 50 years  [Eurocode design level]")
    print("  975-yr RP  =  5% in 50 years")
    print("  2475-yr RP =  2% in 50 years  [collapse prevention level]")

---
## Discussion questions for students

1. **Why does ESHM20 store PoEs as annual rates rather than cumulative probabilities over 50 years?**  
   *Hint: Annual rates are model-independent and can be combined with any exposure window using the Poisson assumption.*

2. **What is the difference between `arithmetic 0.5` and `ordinal 0.5`?**  
   *Hint: Think about a set of five branches weighted equally — the arithmetic mean of their hazard values is not the same as the median branch value.*

3. **Run the discovery for L'Aquila (42.35°N, 13.40°E). Are the same models available? Do the IMTs differ?**

4. **The `rock_vs30_800ms-1` soil type represents a reference rock site (Vs30 = 800 m/s). How would you estimate hazard for a softer site?**  
   *The API only provides rock-reference hazard; site amplification must be applied separately using site factors (e.g., Eurocode 8 soil factors or non-linear amplification functions).*

---
**→ When you are ready, open Notebook 2 to plot hazard curves and response spectra.**